In [ ]:
import os
import json
import torch
import numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM
import torch.nn.functional as F
from tqdm import tqdm
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor
import functools
from collections import defaultdict
import gc
import warnings
warnings.filterwarnings("ignore")

# =====================================================
# Configuration and Data Classes
# =====================================================

@dataclass
class DefenseConfig:
    """Centralized configuration for all defense parameters"""
    # Preemptive defenses
    jpeg_quality: int = 85
    quantization_bits: int = 6
    
    # Prompt smoothing
    num_prompts: int = 3
    prompt_ensemble_weight: float = 1.2
    
    # Noise injection
    noise_level: float = 0.04
    adaptive_noise: bool = True
    
    # Spatial smoothing
    kernel_size: int = 3
    edge_preserve_weight: float = 0.7
    
    # Combined defense
    channel_mix_strength: float = 0.9
    dropout_rate: float = 0.01
    
    # NMS parameters
    iou_threshold: float = 0.45
    
    @classmethod
    def from_attack_strength(cls, epsilon: float) -> 'DefenseConfig':
        """Automatically configure defenses based on attack strength"""
        config = cls()
        
        if epsilon < 0.01:
            # Light defense for weak attacks
            config.jpeg_quality = 90
            config.noise_level = 0.02
            config.kernel_size = 3
        elif epsilon < 0.03:
            # Medium defense
            config.jpeg_quality = 85
            config.noise_level = 0.04
            config.kernel_size = 3
        else:
            # Strong defense for strong attacks
            config.jpeg_quality = 75
            config.noise_level = 0.06
            config.kernel_size = 5
            config.num_prompts = 4
            
        return config

@dataclass
class AttackConfig:
    """Configuration for adversarial attacks"""
    epsilon: float = 0.03
    alpha: float = 0.005
    num_iters: int = 10
    attack_type: str = "pgd"  # "fgsm" or "pgd"

# =====================================================
# Optimized Model Manager
# =====================================================

class Florence2ModelManager:
    """Manages model loading and caching for efficiency"""
    
    def __init__(self, device: str = "cuda", dtype: torch.dtype = torch.float16):
        self.device = torch.device(device)
        self.dtype = dtype
        self.model = None
        self.processor = None
        self._cache = {}
        
    def load_model(self, model_name: str = "microsoft/Florence-2-base", revision: str = "refs/pr/26"):
        """Load model with optimizations"""
        if self.model is None:
            print(f"Loading model on {self.device} with dtype {self.dtype}")
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name, 
                revision=revision,
                torch_dtype=self.dtype,
                trust_remote_code=True
            ).to(self.device)
            
            self.processor = AutoProcessor.from_pretrained(
                model_name,
                revision=revision,
                trust_remote_code=True
            )
            
            # Enable eval mode and gradient checkpointing for memory efficiency
            self.model.eval()
            if hasattr(self.model, 'gradient_checkpointing_enable'):
                self.model.gradient_checkpointing_enable()
                
    def clear_cache(self):
        """Clear GPU cache and internal cache"""
        self._cache.clear()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

# =====================================================
# Optimized Defense Pipeline
# =====================================================

class OptimizedDefensePipeline:
    """Efficient implementation of multi-layered defenses"""
    
    def __init__(self, config: DefenseConfig):
        self.config = config
        self._precomputed = {}
        
    @torch.no_grad()
    def preemptive_defense(self, pil_img: Image.Image) -> Image.Image:
        """Optimized preemptive defense with caching"""
        # Apply JPEG compression
        from io import BytesIO
        buffer = BytesIO()
        pil_img.save(buffer, format="JPEG", quality=self.config.jpeg_quality)
        buffer.seek(0)
        img = Image.open(buffer)
        
        # Quantization
        img_array = np.array(img)
        quant_levels = 2 ** self.config.quantization_bits
        img_array = np.round(img_array * quant_levels / 255) * 255 / quant_levels
        
        return Image.fromarray(img_array.astype(np.uint8))
    
    @torch.no_grad()
    def gaussian_noise_injection(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """Optimized noise injection with adaptive scaling"""
        if self.config.adaptive_noise:
            # Adaptive noise based on image statistics
            mean_val = torch.mean(torch.abs(pixel_values))
            noise_scale = self.config.noise_level * (mean_val / 0.5)
        else:
            noise_scale = self.config.noise_level
            
        # Generate and apply noise efficiently
        noise = torch.randn_like(pixel_values, device=pixel_values.device) * noise_scale
        
        # Frequency-aware mask (precompute if image size is constant)
        h, w = pixel_values.shape[2], pixel_values.shape[3]
        mask_key = (h, w)
        
        if mask_key not in self._precomputed:
            y = torch.linspace(-1, 1, h, device=pixel_values.device)
            x = torch.linspace(-1, 1, w, device=pixel_values.device)
            y_grid, x_grid = torch.meshgrid(y, x, indexing='ij')
            dist = torch.sqrt(x_grid**2 + y_grid**2)
            freq_mask = (1.0 - torch.exp(-2.0 * dist)).unsqueeze(0).unsqueeze(0)
            self._precomputed[mask_key] = freq_mask
        else:
            freq_mask = self._precomputed[mask_key]
            
        return torch.clamp(pixel_values + noise * freq_mask, -2.0, 2.0)
    
    @torch.no_grad()
    def spatial_smoothing(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """Optimized spatial smoothing with edge preservation"""
        # Compute gradients for edge detection
        h_grad = torch.abs(pixel_values[:, :, :, 1:] - pixel_values[:, :, :, :-1])
        v_grad = torch.abs(pixel_values[:, :, 1:, :] - pixel_values[:, :, :-1, :])
        
        # Pad gradients
        h_grad = F.pad(h_grad, (0, 1, 0, 0))
        v_grad = F.pad(v_grad, (0, 0, 0, 1))
        
        grad_mag = (h_grad + v_grad) / 2.0
        edge_mask = torch.exp(-grad_mag * 5.0)
        
        # Apply smoothing
        smoothed = F.avg_pool2d(
            pixel_values,
            kernel_size=self.config.kernel_size,
            stride=1,
            padding=self.config.kernel_size // 2
        )
        
        # Adaptive blending
        blend_weight = self.config.edge_preserve_weight * edge_mask
        return blend_weight * smoothed + (1 - blend_weight) * pixel_values
    
    @torch.no_grad()
    def combined_defense(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """Apply all defenses in optimized sequence"""
        # Apply defenses
        x = self.gaussian_noise_injection(pixel_values)
        x = self.spatial_smoothing(x)
        
        # Quantization
        quant_step = 0.1
        x = torch.round(x / quant_step) * quant_step
        
        # Channel mixing (only for RGB)
        if x.shape[1] == 3:
            b, c, h, w = x.shape
            # Optimized channel mixing
            mix_matrix = torch.eye(3, device=x.device) * self.config.channel_mix_strength
            mix_matrix += (1 - self.config.channel_mix_strength) / 3
            
            x_reshaped = x.view(b, c, -1)
            x_mixed = torch.bmm(mix_matrix.unsqueeze(0).expand(b, -1, -1), x_reshaped)
            x = x_mixed.view(b, c, h, w)
            
            # Dropout
            if self.config.dropout_rate > 0:
                mask = torch.rand_like(x) > self.config.dropout_rate
                x = x * mask
                
        return x

# =====================================================
# Optimized Attack Generation
# =====================================================

class OptimizedAdversarialAttack:
    """Efficient adversarial attack generation"""
    
    def __init__(self, model_manager: Florence2ModelManager):
        self.model_manager = model_manager
        self.attack_cache = {}
        
    @torch.enable_grad()
    def generate_attack(self, pil_img: Image.Image, config: AttackConfig, 
                       defense_pipeline: Optional[OptimizedDefensePipeline] = None) -> Image.Image:
        """Generate adversarial example with optimizations"""
        
        # Apply preemptive defense if pipeline provided
        if defense_pipeline:
            pil_img = defense_pipeline.preemptive_defense(pil_img)
            
        # Prepare inputs
        inputs = self.model_manager.processor(text="<OD>", images=pil_img, return_tensors="pt")
        input_ids = inputs.input_ids.to(self.model_manager.device)
        pixel_values = inputs.pixel_values.to(self.model_manager.device, dtype=self.model_manager.dtype)
        
        if config.attack_type == "fgsm":
            return self._fgsm_attack(pil_img, input_ids, pixel_values, config)
        else:
            return self._pgd_attack(pil_img, input_ids, pixel_values, config)
    
    def _fgsm_attack(self, pil_img: Image.Image, input_ids: torch.Tensor, 
                     pixel_values: torch.Tensor, config: AttackConfig) -> Image.Image:
        """Optimized FGSM attack"""
        pixel_values.requires_grad = True
        
        # Get target IDs
        with torch.no_grad():
            target_ids = self.model_manager.model.generate(
                input_ids=input_ids,
                pixel_values=pixel_values,
                max_new_tokens=256,  # Reduced for speed
                num_beams=3  # Reduced for speed
            )
            
        # Compute gradients
        outputs = self.model_manager.model(
            input_ids=input_ids,
            pixel_values=pixel_values,
            labels=target_ids
        )
        outputs.loss.backward()
        
        # Apply FGSM
        grad_sign = pixel_values.grad.sign()
        adv_pixels = pixel_values + config.epsilon * grad_sign
        adv_pixels = torch.clamp(adv_pixels, -2.0, 2.0)
        
        return self._tensor_to_pil(adv_pixels, pil_img.size)
    
    def _pgd_attack(self, pil_img: Image.Image, input_ids: torch.Tensor,
                    pixel_values: torch.Tensor, config: AttackConfig) -> Image.Image:
        """Optimized PGD attack"""
        orig_pixels = pixel_values.clone()
        
        # Get target IDs once
        with torch.no_grad():
            target_ids = self.model_manager.model.generate(
                input_ids=input_ids,
                pixel_values=pixel_values,
                max_new_tokens=256,
                num_beams=3
            )
        
        # PGD iterations
        for i in range(config.num_iters):
            pixel_values.requires_grad = True
            
            outputs = self.model_manager.model(
                input_ids=input_ids,
                pixel_values=pixel_values,
                labels=target_ids
            )
            
            if outputs.loss is not None:
                outputs.loss.backward()
                
                with torch.no_grad():
                    # Update with gradient sign
                    grad_sign = pixel_values.grad.sign()
                    pixel_values = pixel_values + config.alpha * grad_sign
                    
                    # Project back to epsilon ball
                    delta = torch.clamp(pixel_values - orig_pixels, -config.epsilon, config.epsilon)
                    pixel_values = torch.clamp(orig_pixels + delta, -2.0, 2.0)
                    pixel_values = pixel_values.detach()
                    
        return self._tensor_to_pil(pixel_values, pil_img.size)
    
    def _tensor_to_pil(self, tensor: torch.Tensor, original_size: Tuple[int, int]) -> Image.Image:
        """Convert tensor back to PIL Image efficiently"""
        # Denormalize
        mean = torch.tensor(self.model_manager.processor.image_processor.image_mean,
                          device=tensor.device, dtype=tensor.dtype).view(1, 3, 1, 1)
        std = torch.tensor(self.model_manager.processor.image_processor.image_std,
                         device=tensor.device, dtype=tensor.dtype).view(1, 3, 1, 1)
        
        denorm = (tensor * std) + mean
        denorm = torch.clamp(denorm, 0, 1)
        
        # Convert to PIL
        img_array = (denorm.squeeze(0).permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
        pil_img = Image.fromarray(img_array)
        
        # Resize if needed
        if pil_img.size != original_size:
            pil_img = pil_img.resize(original_size, Image.BICUBIC)
            
        return pil_img

# =====================================================
# Optimized Detection Pipeline
# =====================================================

class OptimizedDetectionPipeline:
    """Efficient object detection with integrated defenses"""
    
    def __init__(self, model_manager: Florence2ModelManager, category_mapping: Dict[str, int]):
        self.model_manager = model_manager
        self.category_mapping = category_mapping
        self.detection_cache = {}
        
    @torch.no_grad()
    def detect_objects(self, pil_img: Image.Image, 
                      defense_pipeline: Optional[OptimizedDefensePipeline] = None,
                      use_prompt_ensemble: bool = False) -> List[Dict]:
        """Run object detection with optional defenses"""
        
        if use_prompt_ensemble and defense_pipeline:
            # Prompt ensemble defense
            return self._prompt_ensemble_detection(pil_img, defense_pipeline)
        
        # Single detection
        return self._single_detection(pil_img, defense_pipeline)
    
    def _single_detection(self, pil_img: Image.Image, 
                         defense_pipeline: Optional[OptimizedDefensePipeline] = None) -> List[Dict]:
        """Single forward pass detection"""
        inputs = self.model_manager.processor(text="<OD>", images=pil_img, return_tensors="pt")
        input_ids = inputs.input_ids.to(self.model_manager.device)
        pixel_values = inputs.pixel_values.to(self.model_manager.device, dtype=self.model_manager.dtype)
        
        # Apply defense if provided
        if defense_pipeline:
            pixel_values = defense_pipeline.combined_defense(pixel_values)
            
        # Generate predictions
        gen_ids = self.model_manager.model.generate(
            input_ids=input_ids,
            pixel_values=pixel_values,
            max_new_tokens=256,
            num_beams=3,
            do_sample=False
        )
        
        # Parse results
        txt = self.model_manager.processor.batch_decode(gen_ids, skip_special_tokens=False)[0]
        parsed = self.model_manager.processor.post_process_generation(
            txt, task="<OD>", image_size=(pil_img.width, pil_img.height)
        ) or {}
        
        od = parsed.get("<OD>", {})
        bboxes = od.get("bboxes", [])
        labels = od.get("labels", [])
        scores = od.get("scores", [1.0] * len(bboxes))
        
        return self._format_detections(bboxes, labels, scores, pil_img.size)
    
    def _prompt_ensemble_detection(self, pil_img: Image.Image, 
                                  defense_pipeline: OptimizedDefensePipeline) -> List[Dict]:
        """Ensemble detection with multiple prompts"""
        prompts = [
            "<OD>",
            "<OD>",  # Double weight
            "<Detect all visible objects>",
            "<Locate and classify all objects>"
        ][:defense_pipeline.config.num_prompts]
        
        all_detections = []
        
        for prompt in prompts:
            inputs = self.model_manager.processor(text=prompt, images=pil_img, return_tensors="pt")
            input_ids = inputs.input_ids.to(self.model_manager.device)
            pixel_values = inputs.pixel_values.to(self.model_manager.device, dtype=self.model_manager.dtype)
            
            # Apply defense
            pixel_values = defense_pipeline.combined_defense(pixel_values)
            
            # Generate
            gen_ids = self.model_manager.model.generate(
                input_ids=input_ids,
                pixel_values=pixel_values,
                max_new_tokens=256,
                num_beams=3
            )
            
            txt = self.model_manager.processor.batch_decode(gen_ids, skip_special_tokens=False)[0]
            parsed = self.model_manager.processor.post_process_generation(
                txt, task="<OD>", image_size=(pil_img.width, pil_img.height)
            ) or {}
            
            od = parsed.get("<OD>", {})
            bboxes = od.get("bboxes", [])
            labels = od.get("labels", [])
            scores = od.get("scores", [1.0] * len(bboxes))
            
            # Boost scores for standard prompt
            if prompt == "<OD>":
                scores = [min(s * defense_pipeline.config.prompt_ensemble_weight, 1.0) for s in scores]
                
            all_detections.extend(list(zip(bboxes, labels, scores)))
        
        # Apply NMS
        if all_detections:
            bboxes, labels, scores = zip(*all_detections)
            bboxes, labels, scores = self._non_max_suppression(
                list(bboxes), list(labels), list(scores), 
                defense_pipeline.config.iou_threshold
            )
            return self._format_detections(bboxes, labels, scores, pil_img.size)
        
        return []
    
    def _non_max_suppression(self, boxes: List, labels: List, scores: List, 
                            iou_threshold: float = 0.5) -> Tuple[List, List, List]:
        """Optimized NMS implementation"""
        if not boxes:
            return [], [], []
            
        # Convert to numpy for efficiency
        boxes_np = np.array(boxes)
        scores_np = np.array(scores)
        
        # Sort by scores
        indices = np.argsort(scores_np)[::-1]
        
        keep = []
        while len(indices) > 0:
            current = indices[0]
            keep.append(current)
            
            if len(indices) == 1:
                break
                
            # Compute IoU with remaining boxes
            current_box = boxes_np[current]
            other_boxes = boxes_np[indices[1:]]
            
            # Vectorized IoU computation
            x1 = np.maximum(current_box[0], other_boxes[:, 0])
            y1 = np.maximum(current_box[1], other_boxes[:, 1])
            x2 = np.minimum(current_box[2], other_boxes[:, 2])
            y2 = np.minimum(current_box[3], other_boxes[:, 3])
            
            intersection = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
            area_current = (current_box[2] - current_box[0]) * (current_box[3] - current_box[1])
            area_others = (other_boxes[:, 2] - other_boxes[:, 0]) * (other_boxes[:, 3] - other_boxes[:, 1])
            union = area_current + area_others - intersection
            
            iou = intersection / (union + 1e-6)
            
            # Keep boxes with different labels or low IoU
            mask = np.array([labels[indices[i+1]] != labels[current] or iou[i] <= iou_threshold 
                           for i in range(len(iou))])
            indices = indices[1:][mask]
            
        # Return filtered results
        keep_boxes = boxes_np[keep].tolist()
        keep_labels = [labels[i] for i in keep]
        keep_scores = scores_np[keep].tolist()
        
        return keep_boxes, keep_labels, keep_scores
    
    def _format_detections(self, bboxes: List, labels: List, scores: List,
                          img_size: Tuple[int, int]) -> List[Dict]:
        """Format detections for COCO evaluation"""
        results = []
        img_w, img_h = img_size
        
        for box, label, score in zip(bboxes, labels, scores):
            if label in self.category_mapping:
                x1, y1, x2, y2 = box
                width = x2 - x1
                height = y2 - y1
                
                # Confidence scoring based on box properties
                box_area = width * height
                img_area = img_w * img_h
                area_ratio = min(box_area / img_area, 0.5)
                
                # Adjust score based on box properties
                adjusted_score = 0.6 + 0.2 * area_ratio + 0.2 * score
                adjusted_score = min(0.98, max(0.6, adjusted_score))
                
                results.append({
                    "bbox": [x1, y1, width, height],
                    "category_id": self.category_mapping[label],
                    "score": adjusted_score
                })
                
        return results

# =====================================================
# Experiment Manager
# =====================================================

class ExperimentManager:
    """Manages experiments with automatic parameter tuning and result tracking"""
    
    def __init__(self, model_manager: Florence2ModelManager, output_dir: str = "./results"):
        self.model_manager = model_manager
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        
        self.results = defaultdict(list)
        self.timing_data = defaultdict(list)
        
    def run_adaptive_evaluation(self, images: List[Tuple[int, Image.Image]], 
                               epsilon_values: List[float] = [0.01, 0.03, 0.05],
                               attack_type: str = "pgd") -> Dict:
        """Run evaluation with adaptive defense selection"""
        
        category_mapping = self._load_category_mapping()
        detection_pipeline = OptimizedDetectionPipeline(self.model_manager, category_mapping)
        attack_generator = OptimizedAdversarialAttack(self.model_manager)
        
        results_by_epsilon = {}
        
        for epsilon in epsilon_values:
            print(f"\n[Evaluating epsilon={epsilon}]")
            
            # Auto-configure defense based on attack strength
            defense_config = DefenseConfig.from_attack_strength(epsilon)
            defense_pipeline = OptimizedDefensePipeline(defense_config)
            attack_config = AttackConfig(epsilon=epsilon, attack_type=attack_type)
            
            # Track results
            clean_results = []
            adv_results = []
            defended_results = []
            
            # Process images with progress bar
            for img_id, img in tqdm(images, desc=f"ε={epsilon}"):
                # Clean detection
                t0 = time.time()
                clean_dets = detection_pipeline.detect_objects(img)
                self.timing_data['clean'].append(time.time() - t0)
                
                for det in clean_dets:
                    det['image_id'] = img_id
                clean_results.extend(clean_dets)
                
                # Adversarial detection (no defense)
                t0 = time.time()
                adv_img = attack_generator.generate_attack(img, attack_config)
                adv_dets = detection_pipeline.detect_objects(adv_img)
                self.timing_data['attack'].append(time.time() - t0)
                
                for det in adv_dets:
                    det['image_id'] = img_id
                adv_results.extend(adv_dets)
                
                # Defended detection
                t0 = time.time()
                adv_img_defended = attack_generator.generate_attack(img, attack_config, defense_pipeline)
                defended_dets = detection_pipeline.detect_objects(
                    adv_img_defended, defense_pipeline, use_prompt_ensemble=True
                )
                self.timing_data['defense'].append(time.time() - t0)
                
                for det in defended_dets:
                    det['image_id'] = img_id
                defended_results.extend(defended_dets)
                
                # Clear cache periodically
                if len(clean_results) % 50 == 0:
                    self.model_manager.clear_cache()
            
            # Save results
            self._save_results(epsilon, attack_type, clean_results, adv_results, defended_results)
            
            # Evaluate
            metrics = self._evaluate_results(epsilon, attack_type)
            results_by_epsilon[epsilon] = metrics
            
            # Print summary
            self._print_summary(epsilon, metrics)
            
        return results_by_epsilon
    
    def generate_report(self, results: Dict) -> None:
        """Generate comprehensive report with visualizations"""
        # Create effectiveness plot
        epsilons = sorted(results.keys())
        clean_aps = [results[e]['clean_ap'] for e in epsilons]
        adv_aps = [results[e]['adv_ap'] for e in epsilons]
        def_aps = [results[e]['defended_ap'] for e in epsilons]
        
        plt.figure(figsize=(10, 6))
        plt.plot(epsilons, clean_aps, 'g-', label='Clean', linewidth=2)
        plt.plot(epsilons, adv_aps, 'r--', label='Adversarial', linewidth=2)
        plt.plot(epsilons, def_aps, 'b-.', label='Defended', linewidth=2)
        plt.xlabel('Attack Strength (ε)', fontsize=12)
        plt.ylabel('mAP', fontsize=12)
        plt.title('Defense Effectiveness Across Attack Strengths', fontsize=14)
        plt.legend(fontsize=11)
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(self.output_dir, 'defense_effectiveness.png'), dpi=300, bbox_inches='tight')
        plt.close()
        
        # Create timing analysis
        avg_times = {k: np.mean(v) for k, v in self.timing_data.items()}
        
        plt.figure(figsize=(8, 6))
        bars = plt.bar(avg_times.keys(), avg_times.values())
        plt.ylabel('Average Time (seconds)', fontsize=12)
        plt.title('Computational Cost Analysis', fontsize=14)
        for bar, time in zip(bars, avg_times.values()):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{time:.3f}s', ha='center', va='bottom')
        plt.savefig(os.path.join(self.output_dir, 'timing_analysis.png'), dpi=300, bbox_inches='tight')
        plt.close()
        
        # Generate text report
        report_path = os.path.join(self.output_dir, 'experiment_report.txt')
        with open(report_path, 'w') as f:
            f.write("Florence-2 Adversarial Robustness Evaluation Report\n")
            f.write("=" * 60 + "\n\n")
            
            for epsilon in sorted(results.keys()):
                metrics = results[epsilon]
                f.write(f"Attack Strength ε = {epsilon}\n")
                f.write("-" * 30 + "\n")
                f.write(f"Clean mAP: {metrics['clean_ap']:.4f}\n")
                f.write(f"Adversarial mAP: {metrics['adv_ap']:.4f}\n")
                f.write(f"Defended mAP: {metrics['defended_ap']:.4f}\n")
                f.write(f"Performance Drop: {metrics['drop']:.4f} ({metrics['drop_percent']:.1f}%)\n")
                f.write(f"Recovery: {metrics['recovery']:.4f} ({metrics['recovery_percent']:.1f}%)\n\n")
            
            f.write("\nComputational Analysis\n")
            f.write("-" * 30 + "\n")
            for k, times in self.timing_data.items():
                f.write(f"{k.capitalize()}: {np.mean(times):.3f}s (±{np.std(times):.3f}s)\n")
        
        print(f"\nReport saved to {report_path}")
    
    def _load_category_mapping(self) -> Dict[str, int]:
        """Load COCO category mapping"""
        annotation_file = "./Dataset/coco/annotations/annotations/instances_val2017.json"
        with open(annotation_file, "r") as f:
            coco_data = json.load(f)
        return {cat["name"]: cat["id"] for cat in coco_data["categories"]}
    
    def _save_results(self, epsilon: float, attack_type: str, 
                     clean_results: List, adv_results: List, defended_results: List):
        """Save results to JSON files"""
        prefix = f"{attack_type}_eps{epsilon}"
        
        with open(os.path.join(self.output_dir, f"{prefix}_clean.json"), "w") as f:
            json.dump(clean_results, f)
        with open(os.path.join(self.output_dir, f"{prefix}_adv.json"), "w") as f:
            json.dump(adv_results, f)
        with open(os.path.join(self.output_dir, f"{prefix}_defended.json"), "w") as f:
            json.dump(defended_results, f)
    
    def _evaluate_results(self, epsilon: float, attack_type: str) -> Dict:
        """Evaluate results using COCO metrics"""
        from pycocotools.coco import COCO
        from pycocotools.cocoeval import COCOeval
        
        ann_file = "./Dataset/coco/annotations/annotations/instances_val2017.json"
        coco_gt = COCO(ann_file)
        
        prefix = f"{attack_type}_eps{epsilon}"
        metrics = {}
        
        # Evaluate clean
        clean_file = os.path.join(self.output_dir, f"{prefix}_clean.json")
        coco_dt = coco_gt.loadRes(clean_file)
        eval_clean = COCOeval(coco_gt, coco_dt, "bbox")
        eval_clean.evaluate()
        eval_clean.accumulate()
        eval_clean.summarize()
        metrics['clean_ap'] = eval_clean.stats[0]
        
        # Evaluate adversarial
        adv_file = os.path.join(self.output_dir, f"{prefix}_adv.json")
        coco_dt = coco_gt.loadRes(adv_file)
        eval_adv = COCOeval(coco_gt, coco_dt, "bbox")
        eval_adv.evaluate()
        eval_adv.accumulate()
        eval_adv.summarize()
        metrics['adv_ap'] = eval_adv.stats[0]
        
        # Evaluate defended
        def_file = os.path.join(self.output_dir, f"{prefix}_defended.json")
        coco_dt = coco_gt.loadRes(def_file)
        eval_def = COCOeval(coco_gt, coco_dt, "bbox")
        eval_def.evaluate()
        eval_def.accumulate()
        eval_def.summarize()
        metrics['defended_ap'] = eval_def.stats[0]
        
        # Calculate effectiveness
        metrics['drop'] = metrics['clean_ap'] - metrics['adv_ap']
        metrics['recovery'] = metrics['defended_ap'] - metrics['adv_ap']
        metrics['drop_percent'] = (metrics['drop'] / metrics['clean_ap'] * 100) if metrics['clean_ap'] > 0 else 0
        metrics['recovery_percent'] = (metrics['recovery'] / metrics['drop'] * 100) if metrics['drop'] > 0 else 0
        
        return metrics
    
    def _print_summary(self, epsilon: float, metrics: Dict):
        """Print evaluation summary"""
        print(f"\n[Summary for ε={epsilon}]")
        print(f"Clean mAP: {metrics['clean_ap']:.4f}")
        print(f"Adversarial mAP: {metrics['adv_ap']:.4f}")
        print(f"Defended mAP: {metrics['defended_ap']:.4f}")
        print(f"Drop: {metrics['drop']:.4f} ({metrics['drop_percent']:.1f}%)")
        print(f"Recovery: {metrics['recovery']:.4f} ({metrics['recovery_percent']:.1f}%)")

# =====================================================
# Main Execution Script
# =====================================================

def main():
    """Main execution function with all optimizations"""
    
    # Configuration
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32
    
    # Image directory
    image_dir = "./Dataset/coco/images/val2017"
    
    # Initialize model manager
    print("Initializing model manager...")
    model_manager = Florence2ModelManager(device=device, dtype=dtype)
    model_manager.load_model()
    
    # Initialize experiment manager
    experiment_manager = ExperimentManager(model_manager)
    
    # Load images
    print("Loading images...")
    images = []
    files = sorted(os.listdir(image_dir))[:100]  # Use subset for testing
    
    for fname in files:
        try:
            img_id = int(os.path.splitext(fname)[0])
            img_path = os.path.join(image_dir, fname)
            img = Image.open(img_path).convert("RGB")
            images.append((img_id, img))
        except:
            continue
    
    print(f"Loaded {len(images)} images")
    
    # Run adaptive evaluation
    print("\nRunning adaptive evaluation...")
    epsilon_values = [0.01, 0.03, 0.05, 0.07]
    
    # Test both attack types
    for attack_type in ["fgsm", "pgd"]:
        print(f"\n{'='*60}")
        print(f"Evaluating {attack_type.upper()} attacks")
        print(f"{'='*60}")
        
        results = experiment_manager.run_adaptive_evaluation(
            images, 
            epsilon_values=epsilon_values,
            attack_type=attack_type
        )
        
        # Generate report
        experiment_manager.generate_report(results)
    
    # Print timing summary
    print("\n[Timing Summary]")
    for operation, times in experiment_manager.timing_data.items():
        avg_time = np.mean(times)
        std_time = np.std(times)
        print(f"{operation}: {avg_time:.3f}s ± {std_time:.3f}s")
    
    print("\nEvaluation complete!")

# =====================================================
# Standalone Functions for Quick Testing
# =====================================================

def quick_test_defenses():
    """Quick test to verify defenses are working"""
    
    # Load a single image
    img_path = "./Dataset/coco/images/val2017/000000000139.jpg"  # Update with actual image
    img = Image.open(img_path).convert("RGB")
    
    # Initialize components
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_manager = Florence2ModelManager(device=device)
    model_manager.load_model()
    
    # Test different epsilon values
    epsilons = [0.01, 0.03, 0.05]
    
    for eps in epsilons:
        print(f"\nTesting ε={eps}")
        
        # Auto-configure defense
        defense_config = DefenseConfig.from_attack_strength(eps)
        defense_pipeline = OptimizedDefensePipeline(defense_config)
        
        # Generate attack
        attack_config = AttackConfig(epsilon=eps, attack_type="fgsm")
        attack_generator = OptimizedAdversarialAttack(model_manager)
        
        # Generate adversarial example
        adv_img = attack_generator.generate_attack(img, attack_config, defense_pipeline)
        
        print(f"Defense config: noise_level={defense_config.noise_level}, "
              f"kernel_size={defense_config.kernel_size}")

def benchmark_performance():
    """Benchmark performance improvements"""
    
    import time
    
    # Initialize
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_manager = Florence2ModelManager(device=device)
    model_manager.load_model()
    
    # Load test image
    img = Image.new("RGB", (640, 480), color="white")
    
    # Configure components
    defense_config = DefenseConfig()
    defense_pipeline = OptimizedDefensePipeline(defense_config)
    detection_pipeline = OptimizedDetectionPipeline(model_manager, {})
    
    # Benchmark operations
    n_runs = 10
    
    print("Benchmarking performance...")
    
    # Detection benchmark
    start = time.time()
    for _ in range(n_runs):
        _ = detection_pipeline.detect_objects(img)
    det_time = (time.time() - start) / n_runs
    print(f"Average detection time: {det_time:.3f}s")
    
    # Defense benchmark
    start = time.time()
    for _ in range(n_runs):
        _ = detection_pipeline.detect_objects(img, defense_pipeline)
    def_time = (time.time() - start) / n_runs
    print(f"Average defended detection time: {def_time:.3f}s")
    print(f"Defense overhead: {(def_time - det_time) / det_time * 100:.1f}%")

if __name__ == "__main__":
    # Run main evaluation
    main()
    
    # Or run quick tests
    # quick_test_defenses()
    # benchmark_performance()